# Options from Zero · Episode 16: Vega, rho, and explaining a day's P&L

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. Vega · 4. Rho · 5. The day · 6. P&L attribution · 7. How big is the residual? · 8. This episode's question · 9. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 16,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

## 2. Last episode's question

Who wants a quiet market? The option **seller**: hedged, they are short gamma and collect theta, so they gain on days the share moves less than the break-even move.

In [ ]:
out["answer"] = {"who": "seller"}

## 3. Vega

**Vega** is the change in price per unit of volatility: $\partial V/\partial\sigma = S\,n(d_1)\sqrt{T}$. Quoted per *vol point* (1%), divide by 100. Like gamma it is largest near the money, but it grows with time to expiry rather than shrinking.

In [ ]:
today = ql.Date(29, 9, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(ql.SimpleQuote(S)), flat(0.0), flat(r),
    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), vol, dc)))
expiry = today + 365   # exactly one year on ACT/365F
g = bs_greeks(S, K, T, r, vol)
opt = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, K), ql.EuropeanExercise(expiry))
opt.setPricingEngine(ql.AnalyticEuropeanEngine(process))
bump_vega = (black_scholes(S, K, T, r, vol + 0.0001) - black_scholes(S, K, T, r, vol - 0.0001)) / 0.0002
assert abs(opt.vega() - g["vega"]) < 1e-9 and abs(bump_vega / g["vega"] - 1) < 1e-6
prices = np.arange(60.0, 150.01, 1.0)
out["vega"] = {"vega": g["vega"], "per_pt": g["vega"] / 100, "quantlib": opt.vega(), "bump": bump_vega,
               "prices": prices.tolist(),
               "one_year": [bs_greeks(s, K, 1.0, r, vol)["vega"] / 100 for s in prices],
               "one_month": [bs_greeks(s, K, 1 / 12, r, vol)["vega"] / 100 for s in prices],
               "one_month_atm": bs_greeks(S, K, 1 / 12, r, vol)["vega"] / 100,
               "ticks": [{"x": float(k), "label": f"${k}"} for k in (60, 75, 90, 105, 120, 135, 150)]}
{k: v for k, v in out["vega"].items() if not isinstance(v, list)}

## 4. Rho

**Rho** is the change in price per unit of interest rate: $\partial V/\partial r = K T e^{-rT} N(d_2)$, per 1% divide by 100. A call gains when rates rise (Episode 4's parity: the strike is paid later, and its present value falls). For a one-month option rho is tiny; for long-dated options, and for interest-rate options (Episode 21), it matters.

In [ ]:
bump_rho = (black_scholes(S, K, T, r + 0.0001, vol) - black_scholes(S, K, T, r - 0.0001, vol)) / 0.0002
assert abs(opt.rho() - g["rho"]) < 1e-9 and abs(bump_rho / g["rho"] - 1) < 1e-6
g1m = bs_greeks(S, K, 1 / 12, r, vol)
out["rho"] = {"rho": g["rho"], "per_pt": g["rho"] / 100, "quantlib": opt.rho(), "one_month_per_pt": g1m["rho"] / 100}
out["rho"]

## 5. The day

Overnight, the share rises from \$100 to \$101.50, the market's volatility for the option rises from 20% to 20.5%, and one day passes. Rates are unchanged.

In [ ]:
S2, vol2, dt = 101.5, 0.205, 1 / 365
new_price = black_scholes(S2, K, T - dt, r, vol2)
actual = new_price - g["price"]
out["day"] = {"S_old": S, "S_new": S2, "vol_old_pct": vol * 100, "vol_new_pct": vol2 * 100, "dS": S2 - S,
              "dvol_pts": (vol2 - vol) * 100, "old_price": g["price"], "new_price": new_price, "actual": actual,
              "rows": [{"what": "Share price", "before": f"${S:.2f}", "after": f"${S2:.2f}"},
                       {"what": "Volatility", "before": f"{vol * 100:.1f}%", "after": f"{vol2 * 100:.1f}%"},
                       {"what": "Time to expiry", "before": "365 days", "after": "364 days"},
                       {"what": "Interest rate", "before": f"{r * 100:.2f}%", "after": f"{r * 100:.2f}%"},
                       {"what": "Call price", "before": f"${g['price']:.3f}", "after": f"${new_price:.3f}"}]}
out["day"]["rows"]

## 6. P&L attribution

Explain the change with the Greeks: $\Delta\cdot dS + \tfrac12\Gamma\cdot dS^2 + \text{vega}\cdot d\sigma + \Theta\cdot dt$. Whatever the Greeks don't explain is the **residual**.

In [ ]:
dS, dv = S2 - S, vol2 - vol
parts = {"delta": g["delta"] * dS, "gamma": 0.5 * g["gamma"] * dS ** 2, "vega": g["vega"] * dv, "theta": g["theta"] * dt}
explained = sum(parts.values())
out["attribution"] = {**parts, "explained": explained, "actual": actual, "residual": actual - explained,
                      "theta_cents_abs": abs(parts["theta"]) * 100,
                      "residual_share": abs(actual - explained) / abs(actual)}
assert abs(out["attribution"]["residual"]) < 0.005
out["attribution"]

## 7. How big is the residual?

The Greeks are a second-order Taylor series (Episode 12) in the share price and first-order in the others, so the residual comes from the terms left out: the share's move cubed, the cross term between share and volatility moves (vanna), vol squared (volga). Scale today's moves up and the residual grows much faster than the moves themselves (it can also change sign, as the left-out terms pull in different directions): the risk numbers are trustworthy for small moves, and need checking against full repricing for big ones.

In [ ]:
grid = []
for k in (0.5, 1, 2, 4):
    s_, v_ = S + dS * k, vol + dv * k
    act = black_scholes(s_, K, T - dt, r, v_) - g["price"]
    exp_ = g["delta"] * dS * k + 0.5 * g["gamma"] * (dS * k) ** 2 + g["vega"] * dv * k + g["theta"] * dt
    grid.append({"scale": k, "dS": dS * k, "dvol_pts": dv * k * 100, "actual": act, "explained": exp_, "residual": act - exp_,
                 "residual_cents_abs": abs(act - exp_) * 100, "share_text": f"+${dS * k:.2f}", "vol_text": f"+{dv * k * 100:.2f} pts"})
out["grid"] = grid
pd.DataFrame(grid)

## 8. This episode's question

Share flat, volatility up one point, one day passes: roughly vega per point plus a day's theta, $+0.399 - 0.015 \approx +0.38$. Exact repricing agrees.

In [ ]:
approx = g["vega"] / 100 + g["theta"] / 365
exact = black_scholes(S, K, T - 1 / 365, r, vol + 0.01) - g["price"]
out["question"] = {"approx": approx, "exact": exact}
assert abs(approx - exact) < 0.01
out["question"]

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")